# 03 · Modelado ALS — Solo juegos (sin DLC ni herramientas)

## Motivación original

El dataset de reviews incluye juegos, DLC y herramientas mezclados. Este notebook explora si **reentrenar
un modelo separado solo con juegos** (excluyendo DLC/herramientas del entrenamiento) mejora las
recomendaciones frente al enfoque del notebook 02 (entrenar con todo el catálogo).

| Enfoque | Hit Rate@10 | MRR@10 |
|---|---|---|
| Popularidad (catálogo solo-juegos) | 0.1225 | 0.0576 |
| ALS entrenado con TODO el catálogo, filtrado a juegos al recomendar | 0.1418 | 0.0589 |

La hipótesis a evaluar en este experimento es que el modelo que contiene unicamente juegos y no DLC/Herramientas es superior al modelo con el catalogo ccompleto.<br>
Para contrastar esta hipótesis, el resultado debe ser significativo ya que implica un sacrificio a nivel de datos (Recomendar un DLC o contenido extra, es perfectamente valido).

In [1]:
import polars as pl
import numpy as np
from scipy.sparse import csr_matrix
from pathlib import Path
import time

DATA_DIR =  Path(r"datasets/parquet/steam/reviews/processed")
GAMES_PATH = Path(r"datasets/parquet/steam/games/games.parquet")

CLEAN_PATH = DATA_DIR / "steam_reviews_clean.parquet"
USER_MAP_PATH = DATA_DIR / "user_map.parquet"
ITEM_MAP_PATH = DATA_DIR / "item_map.parquet"

for p in [CLEAN_PATH, USER_MAP_PATH, ITEM_MAP_PATH, GAMES_PATH]:
    assert p.exists(), f"No encuentro: {p}"

# Referencias del notebook 02, ambas a 100 iteraciones
MODELO1_HR, MODELO1_MRR = 0.1413, 0.0576   # ALS notebook 02, catálogo completo, sin filtrar candidatos
POPULARIDAD_JUEGOS_HR, POPULARIDAD_JUEGOS_MRR = 0.1225, 0.0576   # popularidad, restringida al subset de juegos

print("Todos los archivos existen, OK.")

Todos los archivos existen, OK.


## 1. Inspeccionar `games.parquet`

In [2]:
games_lf = pl.scan_parquet(GAMES_PATH)
games_schema = games_lf.collect_schema()

print(f"Columnas de games.parquet ({len(games_schema)}):\n")
for name, dtype in games_schema.items():
    print(f"  {name:30s} {dtype}")

Columnas de games.parquet (40):

  AppID                          Int64
  Name                           String
  Release date                   String
  Estimated owners               String
  Peak CCU                       Int64
  Required age                   Int64
  Price                          Float64
  Discount                       Int64
  DLC count                      Int64
  About the game                 String
  Supported languages            String
  Full audio languages           String
  Reviews                        String
  Header image                   String
  Website                        String
  Support url                    String
  Support email                  String
  Windows                        Boolean
  Mac                            Boolean
  Linux                          Boolean
  Metacritic score               Int64
  Metacritic url                 String
  User score                     Int64
  Positive                       Int64
  Negative 

In [15]:
GAMES_APPID_COL = "AppID"

assert GAMES_APPID_COL in games_schema, (
    f"No encuentro la columna '{GAMES_APPID_COL}' en games.parquet. "
    f"Columnas disponibles: {list(games_schema.keys())}"
)
print(f"Usando '{GAMES_APPID_COL}' como columna de AppId. OK.")

Usando 'AppID' como columna de AppId. OK.


## 2. Experimento: reentrenar ALS solo con juegos

Filtramos el catálogo y reindexamos desde cero (mismo procedimiento que la primera versión de este
notebook). Esta parte reproduce el experimento documentado en la conclusión de arriba.

In [4]:
item_map = pl.read_parquet(ITEM_MAP_PATH)
print(f"Items totales en el dataset de reviews (juegos + DLC + herramientas): {item_map.height:,}")

juegos_ids = (
    games_lf.select(pl.col(GAMES_APPID_COL).cast(item_map["item_id"].dtype).alias("item_id"))
      .unique()
      .collect()
)

item_map_juegos = item_map.join(juegos_ids, on="item_id", how="semi")
n_descartados = item_map.height - item_map_juegos.height
print(f"\nItems que matchean con games.parquet (juegos): {item_map_juegos.height:,}")
print(f"Items descartados (DLC, herramientas, o no presentes en games.parquet): {n_descartados:,} "
      f"({100*n_descartados/item_map.height:.1f}%)")

Items totales en el dataset de reviews (juegos + DLC + herramientas): 78,376

Items que matchean con games.parquet (juegos): 58,638
Items descartados (DLC, herramientas, o no presentes en games.parquet): 19,738 (25.2%)


In [5]:
item_map_juegos = (
    item_map_juegos
      .with_columns((pl.col("item_id").rank(method="dense").cast(pl.Int32) - 1).alias("item_idx_new"))
      .sort("item_idx_new")
)
n_items = item_map_juegos.height
print(f"n_items (solo juegos): {n_items:,}")

item_idx_translation = item_map_juegos.select(["item_idx", "item_idx_new"])

n_items (solo juegos): 58,638


In [10]:
MIN_REVIEWS_POR_USUARIO_JUEGOS = 2

reviews_lf = pl.scan_parquet(CLEAN_PATH)

reviews_juegos_lf = (
    reviews_lf
      .join(item_idx_translation.lazy(), on="item_idx", how="inner")
      .drop("item_idx")
      .rename({"item_idx_new": "item_idx"})
)

reviews_por_usuario_juegos = reviews_juegos_lf.group_by("user_idx").agg(pl.len().alias("n"))
usuarios_validos_juegos = reviews_por_usuario_juegos.filter(pl.col("n") >= MIN_REVIEWS_POR_USUARIO_JUEGOS).select("user_idx")

CHECKPOINT_JUEGOS = DATA_DIR / "games_only/steam_reviews_juegos.parquet"
(
    reviews_juegos_lf
      .join(usuarios_validos_juegos, on="user_idx", how="semi")
      .sink_parquet(CHECKPOINT_JUEGOS)
)

df_juegos_tmp = pl.scan_parquet(CHECKPOINT_JUEGOS)
n_filas = df_juegos_tmp.select(pl.len()).collect().item()
print(f"Filas (interacciones) en el subset de juegos: {n_filas:,}")

Filas (interacciones) en el subset de juegos: 88,065,480


In [11]:
df_juegos = pl.read_parquet(CHECKPOINT_JUEGOS)

df_juegos = (
    df_juegos
      .with_columns((pl.col("user_idx").rank(method="dense").cast(pl.Int32) - 1).alias("user_idx_new"))
      .drop("user_idx")
      .rename({"user_idx_new": "user_idx"})
)
n_users = df_juegos["user_idx"].max() + 1
print(f"n_users (solo juegos): {n_users:,} | n_items (solo juegos): {n_items:,} | interacciones: {df_juegos.height:,}")

user_idx_translation = (
    pl.read_parquet(CHECKPOINT_JUEGOS)
      .select("user_idx")
      .unique()
      .with_columns((pl.col("user_idx").rank(method="dense").cast(pl.Int32) - 1).alias("user_idx_new"))
)
user_map = pl.read_parquet(USER_MAP_PATH)
user_map_juegos = (
    user_idx_translation.join(user_map, on="user_idx", how="left")
      .drop("user_idx").rename({"user_idx_new": "user_idx"})
      .select(["user_id", "user_idx"]).sort("user_idx")
)
item_map_juegos_final = item_map_juegos.select(["item_id", "item_idx_new"]).rename({"item_idx_new": "item_idx"})

df_juegos.write_parquet(DATA_DIR / "games_only/steam_reviews_juegos.parquet")
user_map_juegos.write_parquet(DATA_DIR / "games_only/user_map.parquet")
item_map_juegos_final.write_parquet(DATA_DIR / "games_only/item_map.parquet")
print("Guardado OK.")

n_users (solo juegos): 16,540,251 | n_items (solo juegos): 58,638 | interacciones: 88,065,480
Guardado OK.


## 3. Train / test split, matriz sparse y entrenamiento

**Hiperparámetros actualizados**: mismos que el Modelo 1 final del notebook 02
(`alpha=32, factors=24, regularization=0.1, iterations=100`) -- para que la comparación sea directa,
usamos exactamente la misma configuración, cambiando únicamente el catálogo de entrenamiento.

In [12]:
df = df_juegos
has_timestamp = "timestamp" in df.columns

if has_timestamp:
    df = df.with_columns(
        pl.col("timestamp").rank(method="ordinal", descending=True).over("user_idx").alias("rank_in_user")
    )
else:
    df = df.with_columns(pl.int_range(pl.len()).shuffle(seed=42).over("user_idx").alias("rank_in_user") + 1)

test_df = df.filter(pl.col("rank_in_user") == 1).drop("rank_in_user")
train_df = df.filter(pl.col("rank_in_user") != 1).drop("rank_in_user")
print(f"Train: {train_df.height:,} filas | Test: {test_df.height:,} filas")

Train: 71,525,229 filas | Test: 16,540,251 filas


In [16]:
ALPHA = 32.0

def build_csr(data: pl.DataFrame, n_users: int, n_items: int, alpha: float = ALPHA) -> csr_matrix:
    pos = data.filter(pl.col("recommended") == 1)
    rows = pos["user_idx"].to_numpy()
    cols = pos["item_idx"].to_numpy()
    vals = np.full(len(pos), 1.0 + alpha, dtype=np.float32)
    return csr_matrix((vals, (rows, cols)), shape=(n_users, n_items))

t0 = time.time()
train_matrix = build_csr(train_df, n_users, n_items)
print(f"Matriz train: {train_matrix.shape}, nnz={train_matrix.nnz:,}, construida en {time.time()-t0:.1f}s")

Matriz train: (16540251, 58638), nnz=61,667,861, construida en 1.9s


In [17]:
# Libreria implicit, Licencia MIT Copyright (c) 2016 Ben Frederickson
# https://github.com/benfred/implicit.git
from implicit.als import AlternatingLeastSquares
from threadpoolctl import threadpool_limits

# Limita BLAS a 1 hilo para evitar conflictos
threadpool_limits(1, "blas")

model = AlternatingLeastSquares(
    factors=24,
    regularization=0.1,
    iterations=100,
    num_threads=0,
    random_state=42,
)

t0 = time.time()
model.fit(train_matrix, show_progress=True)
print(f"\nEntrenamiento completo en {(time.time()-t0)/60:.1f} min")

  0%|          | 0/100 [00:00<?, ?it/s]


Entrenamiento completo en 39.3 min


In [26]:
def evaluate(model, train_matrix, test_df, k=10, sample_users=100000, seed=42):
    test_pos = test_df.filter(pl.col("recommended") == 1)
    users_with_test = test_pos["user_idx"].unique().to_numpy()

    rng = np.random.default_rng(seed)
    if len(users_with_test) > sample_users:
        users_with_test = rng.choice(users_with_test, size=sample_users, replace=False)

    test_lookup = dict(zip(test_pos["user_idx"].to_list(), test_pos["item_idx"].to_list()))

    ids, _ = model.recommend(
        users_with_test, train_matrix[users_with_test], N=k, filter_already_liked_items=True,
    )

    hits, reciprocal_ranks = 0, []
    for row, uidx in enumerate(users_with_test):
        true_item = test_lookup[uidx]
        recs = ids[row]
        if true_item in recs:
            hits += 1
            rank = int(np.where(recs == true_item)[0][0]) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    return hits / len(users_with_test), float(np.mean(reciprocal_ranks)), len(users_with_test)

hr_reentrenado, mrr_reentrenado, n_eval = evaluate(model, train_matrix, test_df, k=10)
print(f"ALS reentrenado (solo juegos, 100 iter) \nHit Rate@10: {hr_reentrenado:.4f} \nMRR@10: {mrr_reentrenado:.4f} (n={n_eval:,})")

ALS reentrenado (solo juegos, 100 iter) 
Hit Rate@10: 0.1426 
MRR@10: 0.0582 (n=100,000)


## 4. Baseline de popularidad (sobre este mismo subset de juegos)

In [27]:
top_items_pop = (
    train_df.filter(pl.col("recommended") == 1)
      .group_by("item_idx").agg(pl.len().alias("n"))
      .sort("n", descending=True).head(10)["item_idx"].to_numpy()
)

test_pos = test_df.filter(pl.col("recommended") == 1)
test_lookup = dict(zip(test_pos["user_idx"].to_list(), test_pos["item_idx"].to_list()))

hits_pop, rr_pop = 0, []
for uidx, true_item in test_lookup.items():
    if true_item in top_items_pop:
        hits_pop += 1
        rank = int(np.where(top_items_pop == true_item)[0][0]) + 1
        rr_pop.append(1.0 / rank)
    else:
        rr_pop.append(0.0)

hr_pop_juegos = hits_pop / len(test_lookup)
mrr_pop_juegos = float(np.mean(rr_pop))
print(f"Popularidad (solo juegos) — Hit Rate@10: {hr_pop_juegos:.4f} | MRR@10: {mrr_pop_juegos:.4f}")

Popularidad (solo juegos) — Hit Rate@10: 0.1225 | MRR@10: 0.0576


## 5. Comparación: ¿vale la pena reentrenar?

Este es el resultado que determina la conclusión de la introducción -- comparamos el modelo reentrenado de
este notebook contra popularidad. El tercer punto de comparación (filtrado al servir, sin reentrenar) se
calcula en la sección 8, usando el modelo del notebook 02.

In [28]:
comparacion = pl.DataFrame([
    {"enfoque": "Popularidad (solo juegos)",       "hit_rate": hr_pop_juegos,     "mrr": mrr_pop_juegos},
    {"enfoque": "ALS reentrenado (solo juegos)",   "hit_rate": hr_reentrenado,    "mrr": mrr_reentrenado},
])
comparacion

enfoque,hit_rate,mrr
str,f64,f64
"""Popularidad (solo juegos)""",0.122547,0.057638
"""ALS reentrenado (solo juegos)""",0.14257,0.058209


## 6. Recomendaciones de ejemplo (modelo reentrenado)

Incluye el mismo filtro de piso mínimo de reviews para `similar_items` que en el notebook 04, dado que la
causa raíz (ítems de cola larga con similitud espuria) es la misma independientemente del catálogo usado
para entrenar.

In [29]:
item_lookup = dict(zip(item_map_juegos_final["item_idx"].to_list(), item_map_juegos_final["item_id"].to_list()))
user_lookup = dict(zip(user_map_juegos["user_idx"].to_list(), user_map_juegos["user_id"].to_list()))
user_idx_lookup = {v: k for k, v in user_lookup.items()}
item_idx_lookup = {v: k for k, v in item_lookup.items()}

TARGET_USER_ID = 76561198324251650

if TARGET_USER_ID not in user_idx_lookup:
    print(f"El usuario {TARGET_USER_ID} no está en el subset de juegos.")
else:
    sample_user_idx = user_idx_lookup[TARGET_USER_ID]
    ids, scores = model.recommend(sample_user_idx, train_matrix[sample_user_idx], N=10)
    print(f"Recomendaciones (solo juegos, reentrenado) para usuario {TARGET_USER_ID}:\n")
    for item_idx, score in zip(ids, scores):
        print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}")

Recomendaciones (solo juegos, reentrenado) para usuario 76561198324251650:

  app_id=601150       score=0.4465
  app_id=698780       score=0.4240
  app_id=814380       score=0.4028
  app_id=678960       score=0.3489
  app_id=335300       score=0.3345
  app_id=227300       score=0.3326
  app_id=230410       score=0.3299
  app_id=1245620      score=0.3194
  app_id=637650       score=0.3145
  app_id=570940       score=0.3142


In [30]:
MIN_REVIEWS_PARA_SIMILITUD = 100

n_reviews_por_item = (
    pl.concat([train_df.select(["item_idx"]), test_df.select(["item_idx"])])
      .group_by("item_idx").agg(pl.len().alias("n_reviews"))
)
n_reviews_lookup = dict(zip(n_reviews_por_item["item_idx"].to_list(), n_reviews_por_item["n_reviews"].to_list()))

TARGET_APP_ID = 489830

if TARGET_APP_ID not in item_idx_lookup:
    print(f"El app_id={TARGET_APP_ID} no está en el subset de juegos.")
else:
    sample_item_idx = item_idx_lookup[TARGET_APP_ID]
    similar_ids, similar_scores = model.similar_items(sample_item_idx, N=30)

    print(f"Juegos similares (solo juegos, reentrenado) a app_id={TARGET_APP_ID}:\n")
    mostrados = 0
    for item_idx, score in zip(similar_ids, similar_scores):
        n_rev = n_reviews_lookup.get(int(item_idx), 0)
        if n_rev < MIN_REVIEWS_PARA_SIMILITUD:
            continue
        print(f"  app_id={item_lookup[int(item_idx)]:<12} score={score:.4f}  (n_reviews={n_rev:,})")
        mostrados += 1
        if mostrados == 10:
            break

Juegos similares (solo juegos, reentrenado) a app_id=489830:

  app_id=489830       score=1.0000  (n_reviews=230,733)
  app_id=377160       score=0.8560  (n_reviews=283,779)
  app_id=72850        score=0.7765  (n_reviews=260,094)
  app_id=22380        score=0.7753  (n_reviews=164,693)
  app_id=306130       score=0.7594  (n_reviews=113,786)
  app_id=900883       score=0.7494  (n_reviews=42,656)
  app_id=22330        score=0.7488  (n_reviews=42,656)
  app_id=1151340      score=0.7220  (n_reviews=81,189)
  app_id=22370        score=0.6509  (n_reviews=37,423)
  app_id=22320        score=0.6102  (n_reviews=20,345)


## 7. Guardar el modelo reentrenado

Se almacena en directorio models/Tunned para futuras referencias

In [31]:
MODEL_DIR = Path(r"models/Tunned")
MODEL_DIR.mkdir(exist_ok=True)
model.save(str(MODEL_DIR / "als_model_games.npz"))
print("Modelo guardado en:", MODEL_DIR / "als_model_games.npz")

Modelo guardado en: models/Tunned/als_model_games.npz


In [32]:
# Libreria implicit, Licencia MIT Copyright (c) 2016 Ben Frederickson
# https://github.com/benfred/implicit.git
from implicit.cpu.als import AlternatingLeastSquares
from threadpoolctl import threadpool_limits

# Limita BLAS a 1 hilo para evitar conflictos
threadpool_limits(1, "blas")

MODEL_DIR = Path(r"models/Tunned")
model = AlternatingLeastSquares.load(str(MODEL_DIR / "als_model_games.npz"))

## 8. Contraste de hipótesis (95% de confianza): ¿reentrenar aporta una diferencia real?

**H₀**: no hay diferencia real entre los domodelos — **H₁**existe diferencia significativa que apoya utilizar un modelo sobre otroay.

### Hit Rate@10 (test de dos proporciones)

- ALS reentrenado (solo juegos): p₁ = 0.1426
- ALS completo, filtrado al servir: p₂ = 0.1418
- n₁ = n₂ = 100,000

$$SE = \sqrt{2 \cdot \bar{p}(1-\bar{p})/n} \approx 0.00156$$
$$z = \frac{0.1426 - 0.1418}{0.00156} \approx 0.51$$

**p-valor (dos colas) ≈ 0.61** — muy por encima de 0.05.

### MRR@10 (test de dos medias)

Usando la desviación estándar estimada de la distribución de reciprocal-rank (≈0.18, a partir del análisis de distribución de ranks del notebook 02):

$$SE_{diff} \approx \sqrt{2} \cdot \frac{0.18}{\sqrt{100000}} \approx 0.00081$$
$$z = \frac{0.0589 - 0.0582}{0.00081} \approx 0.86$$

**p-valor (dos colas) ≈ 0.39** — también muy por encima de 0.05.

### Conclusión

**No se rechaza H₀ en ninguna de las dos métricas.** Con 95% de confianza, la diferencia observada entre reentrenar un modelo específico para "solo juegos" y simplemente filtrar los candidatos de un modelo entrenado con todo el catálogo **no es estadísticamente distinguible de cero** — ambos debajo del opción preferible.

## 9. Resumen final

| Enfoque | Hit Rate@10 | MRR@10 | Mantenimiento en producción |
|---|---|---|---|
| Popularidad (solo juegos) | 0.1225 | 0.0576 | Trivial |
| ALS reentrenado con solo juegos | 0.1426 | 0.0582 | Un modelo extra para reentrenar por cada recorte de catálogo |
| **ALS único (todo el catálogo) + filtro al recomendar** | 0.1418 | 0.0589 | **Un solo modelo, cero mantenimiento extra** |

**Ganador**: el enfoque de un solo modelo con filtrado de candidatos al servir. Mismo resultado
estadísticamente, menor costo operativo (no se requieren dos modelos separados en caso de querer recomendar DLCs).